# A real-world agentic RAG application, evaluated with AuditKIT

The system under test is built the way teams actually build these:
- **LangGraph** `create_react_agent`: a standard agent framework running its own tool loop.
- **A real corpus**: `rag-datasets/rag-mini-wikipedia` on the Hugging Face Hub, with 3,200 Wikipedia passages and 900+ human-written question/answer pairs.
- **Dense retrieval**: `sentence-transformers/all-MiniLM-L6-v2` embeddings, top-k by cosine similarity.
- **A hosted LLM** behind an OpenAI-compatible API: Cohere, Hugging Face Inference Providers, or a vLLM server on this GPU.
- **Served over HTTP**, like a production service. AuditKIT evaluates it as a black box through `agent:<url>`.

It then walks the loop a team runs before shipping:

| § | Step | AuditKIT |
|---|---|---|
| 1 | Build the agent and serve it | – |
| 2 | Evaluate on real questions | `agent:` with tool, retrieval, grounding and correctness metrics |
| 3 | Error analysis: *why* did it fail? | per-question scores, bucketed as retrieval miss, ungrounded or wrong |
| 4 | A/B test a change (retrieval `top_k` 2 vs 5) | `compare_models` with paired significance |
| 5 | Score production logs without re-running | `episode_from_openai_messages`, then `rescore` |

**Backends** (`BACKEND`):

| Value | What it runs | Needs |
|---|---|---|
| `"cohere"` | Command A | `CO_API_KEY` |
| `"hf"` | HF Inference Providers | `HF_TOKEN` |
| `"vllm"` | Qwen3 served on this runtime's GPU (Colab L4/A100) | nothing |
| `"openai"` | any OpenAI-compatible URL | `NB_BASE_URL`, `NB_MODEL` |

The judge uses the same endpoint.

**Secrets:** `GITHUB_TOKEN` (to install AuditKIT), plus the backend's key.

**Cost:** about 20 questions × (a few agent calls + about 8 judge calls), and §4 doubles the agent part.

In [ ]:
# ---- settings ----
import os
BACKEND = os.environ.get("NB_BACKEND", "cohere")        # "cohere" | "hf" | "vllm" | "openai"
N_QUESTIONS = int(os.environ.get("NB_N", "20"))
TOP_K = 3
RUN_AB = os.environ.get("NB_RUN_AB", "1") == "1"         # §4 re-runs the agent with top_k 2 and 5
ENDPOINTS = {
    "cohere": ("https://api.cohere.ai/compatibility/v1", "command-a-03-2025", "CO_API_KEY"),
    "hf": ("https://router.huggingface.co/v1", "Qwen/Qwen2.5-72B-Instruct", "HF_TOKEN"),
    "vllm": ("http://127.0.0.1:8000/v1", "Qwen/Qwen3-4B", None),
    "openai": (os.environ.get("NB_BASE_URL", ""), os.environ.get("NB_MODEL", ""), None),
}

In [ ]:
# ---- install (Colab) / reuse (local checkout) ----
# Colab: clones AuditKIT at BRANCH (#1's branch, where #10 and every #15 fix are merged) and installs it.
# Locally: uses the auditkit already importable (e.g. the repo's .venv kernel).
import os, subprocess, sys
BRANCH = "feat/rag-agent-evals"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    for key in ('GITHUB_TOKEN', 'HF_TOKEN', 'CO_API_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass                          # secret not set / not granted to this notebook
    repo_url = f"https://{os.environ['GITHUB_TOKEN']}@github.com/aryapratinavseth/AuditKIT-Internal.git"
    if not os.path.isdir("/content/AuditKIT-Internal"):
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, repo_url, "/content/AuditKIT-Internal"], check=True)
    else:   # a reused runtime: move the existing clone to the branch head, never run stale code
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "fetch", "-q", repo_url, BRANCH], check=True)
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e",
                    "/content/AuditKIT-Internal[requests]"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "langgraph", "langchain-openai",
                    "sentence-transformers", "datasets"], check=True)
    sys.path.insert(0, "/content/AuditKIT-Internal/src")
import auditkit as ak
print("auditkit", ak.__version__, "| secrets:", {k: bool(os.environ.get(k)) for k in ('HF_TOKEN', 'CO_API_KEY')})

In [ ]:
import json, re, subprocess, sys, threading, time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import numpy as np
import pandas as pd
import requests
from auditkit import Sample
from auditkit.runspec import RunConfig
pd.set_option("display.max_colwidth", 70)

BASE_URL, MODEL, KEY_ENV = ENDPOINTS[BACKEND]
API_KEY = os.environ.get(KEY_ENV, "") if KEY_ENV else os.environ.get("NB_API_KEY", "EMPTY")
assert BASE_URL and MODEL, "set NB_BASE_URL / NB_MODEL for BACKEND='openai'"
assert API_KEY, f"BACKEND={BACKEND!r} needs {KEY_ENV}"
THINKING = "Qwen3" in MODEL and "Instruct-2507" not in MODEL   # Qwen3 hybrid models think unless told not to

if BACKEND == "vllm":        # serve the model on this GPU with a tool-call parser
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "vllm>=0.30"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)
    server_log = open("/tmp/vllm.log", "w")
    vllm_proc = subprocess.Popen([sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, "--port", "8000",
                                  "--max-model-len", "8192", "--enable-auto-tool-choice", "--tool-call-parser", "hermes"],
                                 stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(90):
        try:
            if requests.get("http://127.0.0.1:8000/health", timeout=5).ok:
                break
        except requests.RequestException:
            time.sleep(10)
    else:
        raise RuntimeError("vLLM did not start:\n" + open("/tmp/vllm.log").read()[-3000:])
print("LLM:", BACKEND, MODEL, "@", BASE_URL)

## 1 · The application

### Corpus and retriever
The passages are embedded once. `retrieve()` returns the top-k passages by cosine similarity.

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
corpus = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus", split="passages")
qa = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer", split="test")
PASSAGES = {str(r["id"]): r["passage"].strip() for r in corpus if r["passage"].strip()}
IDS = list(PASSAGES)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
EMB = embedder.encode([PASSAGES[i] for i in IDS], batch_size=128, normalize_embeddings=True, show_progress_bar=False)

def retrieve(query, k):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    top = np.argsort(-EMB @ q)[:k]
    return [(IDS[i], PASSAGES[IDS[i]], float(EMB[i] @ q)) for i in top]

print(len(PASSAGES), "passages |", len(qa), "questions")
print(retrieve("Who was the 16th president of the United States?", 2)[0][1][:200])

### The LangGraph agent
A ReAct agent with one tool, `search_wikipedia`. The service records what each request retrieved, as a production RAG service logs its sources. `top_k` is a per-request setting, so §4 can A/B test it.

In [ ]:
from langchain_core.messages import convert_to_openai_messages
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# per-request state; contextvars (unlike threading.local) follow the call into LangGraph's tool executor threads
from contextvars import ContextVar
_retrieved, _top_k = ContextVar("retrieved"), ContextVar("top_k", default=TOP_K)

@tool
def search_wikipedia(query: str) -> str:
    """Search the Wikipedia knowledge base. Returns the most relevant passages, each with its id."""
    hits = retrieve(query, _top_k.get())
    _retrieved.get().extend(hits)
    return "\n\n".join(f"[{pid}] {text}" for pid, text, _ in hits)

SYSTEM = ("You answer questions using the Wikipedia knowledge base. Always call search_wikipedia first; search again with a "
          "different query if the passages don't contain the answer. Answer ONLY from the passages, in one or two sentences. "
          "If they don't contain the answer, say you don't know.")
llm = ChatOpenAI(model=MODEL, base_url=BASE_URL, api_key=API_KEY, temperature=0, max_tokens=512, timeout=180, max_retries=6,
                 **({"extra_body": {"chat_template_kwargs": {"enable_thinking": False}}} if THINKING else {}))
import warnings
with warnings.catch_warnings():            # LangGraph 1.x: create_react_agent works, and is slated to move to langchain.agents in 2.0
    warnings.simplefilter("ignore")
    agent = create_react_agent(llm, [search_wikipedia], prompt=SYSTEM)
TOOL_SCHEMA = [convert_to_openai_tool(search_wikipedia)]

def answer(question, top_k=TOP_K):
    log = []
    _retrieved.set(log); _top_k.set(top_k)
    out = agent.invoke({"messages": [("user", question)]}, {"recursion_limit": 12})
    msgs = convert_to_openai_messages(out["messages"])
    final = next((m.get("content") or "" for m in reversed(msgs) if m["role"] == "assistant" and not m.get("tool_calls")), "")
    seen, contexts = set(), []
    for pid, text, _ in log:                        # ranked, de-duplicated across searches
        if pid not in seen:
            seen.add(pid); contexts.append(f"[{pid}] {text}")
    return {"output": final, "messages": msgs, "contexts": contexts}

demo = answer("Who was the 16th president of the United States?")
print(demo["output"]); print("sources:", [c.split("]")[0] + "]" for c in demo["contexts"]])

### Serve it
It's a plain HTTP service: POST `{"input": ...}` and get back `output`, the OpenAI-format `messages` transcript, and the `contexts` it used. That's the contract AuditKIT's `agent:` backend reads. An optional `top_k` field in the request body configures retrieval.

In [ ]:
TRANSCRIPTS = {}          # "production log": question -> transcript, for §5

class Service(BaseHTTPRequestHandler):
    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        try:
            res, code = answer(body["input"], int(body.get("top_k", TOP_K))), 200
            TRANSCRIPTS[(body["input"], int(body.get("top_k", TOP_K)))] = res
        except Exception as e:  # noqa: BLE001 -- report it to the caller
            res, code = {"error": f"{type(e).__name__}: {e}"}, 500
        data = json.dumps(res).encode()
        self.send_response(code); self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data))); self.end_headers(); self.wfile.write(data)
    def log_message(self, *a):
        pass

httpd = ThreadingHTTPServer(("127.0.0.1", 0), Service)
threading.Thread(target=httpd.serve_forever, daemon=True).start()
SERVICE = f"http://127.0.0.1:{httpd.server_address[1]}/answer"
print("service at", SERVICE)

## 2 · Evaluate it on real questions

A seeded sample of the dataset's questions, each with its human-written answer as `target`.

The dataset has no gold passage labels. So `reference_contexts` is a **silver** label: the passages that contain the gold answer string. It's set only for answers of at least 4 characters that appear in the corpus, not for "yes" or "no". `RetrievalMetrics` uses it; the reference-free judges don't need it.

| Group | Metrics |
|---|---|
| Correctness | `f1_score` against the gold answer, `task_completion` (judge) |
| Grounding | `faithfulness`, `response_groundedness`, `hallucination` (lower is better), `answer_relevancy` |
| Retrieval | `context_relevance` and `context_recall` (judge), `retrieval` against the silver labels (@3) |
| Agent behaviour | searched at least once (`tool_call_f1` by name), `redundant_tool_calls`, `agent_loop_detection`, `tool_call_validity` |

Two notes:
- `f1_score` is low when the agent answers in a full sentence and the gold is "Yes", so read `task_completion` as the correctness signal.
- The warning "*Samples carry tools but the adapter is 'generation'*" is expected. The service owns its tool, and `tools` is on the samples only so `tool_call_validity` can check calls against the schema.

In [ ]:
rng = np.random.default_rng(0)
rows = [qa[int(i)] for i in rng.choice(len(qa), size=N_QUESTIONS, replace=False)]
def silver(answer_text):
    a = answer_text.strip().lower()
    if len(a) < 4 or a in ("yes", "no"):
        return None
    hits = [f"[{pid}] {t}" for pid, t in PASSAGES.items() if a in t.lower()]
    return hits or None
SEARCH = [[{"name": "search_wikipedia", "arguments": {}}]]          # scored by name (arg_mode="subset")
EVAL = [Sample(id=f"q{r['id']}", input=r["question"], target=r["answer"], tools=TOOL_SCHEMA,
               expected_tool_calls=SEARCH, reference_contexts=silver(r["answer"])) for r in rows]
print(sum(s.reference_contexts is not None for s in EVAL), "of", len(EVAL), "questions have silver passages")

In [ ]:
from auditkit.model.api_gen import APIModel
from auditkit.model import Model

_judge = APIModel(MODEL, api_base=BASE_URL, api_key=API_KEY, name=f"judge:{MODEL}")
_judge.max_retries, _judge.retry_delay = 6, 10
JUDGE = _judge
# A thinking model as the judge never writes a verdict; the judges build their own requests, so
# the switch is theirs (judge_chat_template_kwargs), not the run's.
J = {"judge_model": JUDGE, **({"judge_chat_template_kwargs": {"enable_thinking": False}} if THINKING else {})}
SCORERS = [
    "f1_score", ak.TaskCompletion(**J),
    ak.Faithfulness(**J), ak.ResponseGroundedness(**J), ak.Hallucination(**J), ak.AnswerRelevancy(**J),
    ak.ContextRelevance(**J), ak.ContextRecall(**J), ak.RetrievalMetrics(k=3),
    ak.ToolCallF1(arg_mode="subset"), ak.RedundantToolCalls(), ak.AgentLoopDetection(), ak.ToolCallValidity(),
]
CFG = RunConfig(timeout=600, concurrency=1)
run = ak.evaluate(EVAL, model=f"agent:{SERVICE}", scorers=SCORERS, config=CFG)
GROUPS = {"correctness": ("f1_score", "task_completion"),
          "grounding": ("faithfulness", "response_groundedness", "hallucination", "answer_relevancy"),
          "retrieval": ("context_relevance", "context_recall", "hit_rate@3", "recall@3", "mrr@3"),
          "agent": ("tool_call_recall_subset", "tool_call_precision_subset", "redundant_tool_calls", "agent_loop_detection", "tool_call_validity")}
for g, keys in GROUPS.items():
    print(f"{g:12s}", {k: round(run.headline[k], 3) for k in keys if k in run.headline})
print("judge errors (unreadable verdicts, never scored 0):", len(run.errors), "| unscored:", run.unscored)

## 3 · Error analysis

The averages say *how well*, and the per-question scores say *why*. Each question is bucketed by the first thing that went wrong:
- **answered without searching**: it skipped the tool and answered from the model's own memory, which is ungrounded by construction;
- **retrieval miss**: the silver passage wasn't retrieved;
- **not grounded**: the answer isn't supported by what was retrieved;
- **wrong answer**: grounded, but the judge says the task wasn't completed;
- **ok**: none of the above.

In [ ]:
def scores(p):
    return {s["name"]: s["value"] for s in (p.metadata or {}).get("scores", []) if s.get("value") is not None}
table = []
for p, s in zip(run.predictions, EVAL):
    sc = scores(p)
    if sc.get("tool_call_recall_subset") == 0:
        bucket = "answered without searching"
    elif sc.get("recall@3") == 0:
        bucket = "retrieval miss"
    elif sc.get("faithfulness", 1) < 0.5 or sc.get("response_groundedness", 1) < 0.5:
        bucket = "not grounded"
    elif sc.get("task_completion", 1) < 1:
        bucket = "wrong answer"
    else:
        bucket = "ok"
    table.append({"id": p.sample_id, "question": s.input[:60], "gold": s.target[:30], "answer": (p.raw_output or "")[:70],
                  "bucket": bucket, **{k: round(sc[k], 2) for k in ("f1_score", "task_completion", "faithfulness", "recall@3") if k in sc}})
df = pd.DataFrame(table)
display(df["bucket"].value_counts().to_frame("questions"))
display(df.sort_values("bucket"))

## 4 · A/B test a change: retrieval `top_k` 2 vs 5

The team wants to know whether retrieving more passages helps. The same service is evaluated with each setting; `extra_body` adds `top_k` to every request body. `compare_models` lines the runs up per metric, and `significance()` runs a paired test on the per-question scores, so the decision isn't made on noise.

In [ ]:
if RUN_AB:
    from auditkit.model.agent_endpoint import AgentEndpointModel
    arms = {f"top_k={k}": AgentEndpointModel(url=SERVICE, name=f"agent:top_k={k}", extra_body={"top_k": k}, timeout=600) for k in (2, 5)}
    AB_SCORERS = ["f1_score", ak.TaskCompletion(**J), ak.Faithfulness(**J), ak.ContextRecall(**J), ak.RetrievalMetrics(k=3)]
    ab = ak.compare_models(list(arms.values()), EVAL, scorers=AB_SCORERS, model_names=list(arms), config=CFG)
    display(pd.DataFrame(ab.comparison_table()))
    for metric in ("task_completion", "context_recall", "f1_score"):
        try:
            sig = ab.significance("top_k=2", "top_k=5", metric=metric)
            print(metric, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in sig.items() if not isinstance(v, (list, dict))})
        except Exception as e:  # noqa: BLE001 -- too few paired scores for this metric
            print(metric, "no paired test:", e)
else:
    print("skipped (RUN_AB=0)")

## 5 · Score production logs without re-running the agent

In production you usually have transcripts, not a test harness. `episode_from_openai_messages` turns a logged OpenAI-format conversation into an `AgentEpisode`, the same object the agent-eval runner uses. `rescore` then applies metrics offline: no model calls and no agent re-run. Here the service's own log from §2 stands in for production traffic.

In [ ]:
from auditkit.agent_eval import AgentCase, episode_from_openai_messages, rescore
episodes, cases = [], []
for s in EVAL:
    t = TRANSCRIPTS.get((s.input, TOP_K))
    if not t:
        continue
    ep = episode_from_openai_messages(t["messages"], case_id=s.id, final_answer=t["output"], retrieved_contexts=t["contexts"])
    episodes.append(ep)
    cases.append(AgentCase(id=s.id, task=s.input, reference_turns=SEARCH[0]))
offline = rescore(episodes, ["tool_call_f1_name", "redundant_tool_calls", "agent_loop_detection"], cases=cases)
print(len(episodes), "logged episodes rescored offline")
display(pd.DataFrame([{"case": r.case_id,
                       **{k: round(v["value"], 3) for k, v in (r.diagnostics or {}).items() if k in ("tool_call_recall_name", "redundant_tool_calls", "agent_loop_detection")},
                       "not scored (why)": r.ineligible or ""} for r in offline.rows]))

## Summary

In [ ]:
summary = {k: round(v, 3) for k, v in run.headline.items() if any(k in keys for keys in GROUPS.values())}
display(pd.Series(summary, name=f"{BACKEND}:{MODEL} (top_k={TOP_K})").to_frame())
print("\nbuckets:", df["bucket"].value_counts().to_dict())

### What this shows
- **One black-box contract** (`agent:<url>`) covers a LangGraph app, or any service that returns its answer, transcript and sources.
- **Metrics that point at the failing stage:**
  - retrieval: `recall@3`, `context_recall`;
  - grounding: `faithfulness`, `hallucination`;
  - correctness: `task_completion`, `f1_score`;
  - agent behaviour: redundant searches, loops.
- **Judges fail loudly:** an unreadable judge verdict is an error, never a silent 0.
- **Changes are decided with paired tests** (§4), not eyeballed averages.
- **Logged transcripts are scored offline** (§5), so regressions can be tracked on real traffic.